In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from collections import Counter
import os
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [19]:
# ── Load the CIC-IDS 2018 combined CSV ───────────────────────────────
df = pd.read_csv("cicids2018_SMALL.csv", low_memory=False)

print(f"Raw dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['Label'].value_counts()}")

Raw dataset shape: (2710220, 80)

Columns: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'F

In [20]:
# Remove any stray header rows that appear as data
df = df[df["Label"] != "Label"]
df.reset_index(drop=True, inplace=True)
print(f"After removing stray header rows: {df.shape}")
print(df["Label"].value_counts())

After removing stray header rows: (2710220, 80)
Label
Benign                      1753356
DoS attacks-Hulk             461912
Bot                          286191
DoS attacks-SlowHTTPTest     139890
Infilteration                 68871
Name: count, dtype: int64


In [21]:
# Drop Timestamp (non-numeric, not a flow feature) and Dst Port (identifier)
cols_to_drop = ["Timestamp", "Dst Port"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print(f"Dropped columns: {[c for c in cols_to_drop if c in df.columns]}")
print(f"Shape after drop: {df.shape}")

Dropped columns: []
Shape after drop: (2710220, 78)


In [22]:
# Convert all feature columns to numeric (Label stays as string)
feature_cols = [c for c in df.columns if c != "Label"]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
print("Converted all feature columns to numeric.")
df.info()

Converted all feature columns to numeric.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2710220 entries, 0 to 2710219
Data columns (total 78 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Protocol           int64  
 1   Flow Duration      int64  
 2   Tot Fwd Pkts       int64  
 3   Tot Bwd Pkts       int64  
 4   TotLen Fwd Pkts    int64  
 5   TotLen Bwd Pkts    float64
 6   Fwd Pkt Len Max    int64  
 7   Fwd Pkt Len Min    int64  
 8   Fwd Pkt Len Mean   float64
 9   Fwd Pkt Len Std    float64
 10  Bwd Pkt Len Max    int64  
 11  Bwd Pkt Len Min    int64  
 12  Bwd Pkt Len Mean   float64
 13  Bwd Pkt Len Std    float64
 14  Flow Byts/s        float64
 15  Flow Pkts/s        float64
 16  Flow IAT Mean      float64
 17  Flow IAT Std       float64
 18  Flow IAT Max       float64
 19  Flow IAT Min       float64
 20  Fwd IAT Tot        float64
 21  Fwd IAT Mean       float64
 22  Fwd IAT Std        float64
 23  Fwd IAT Max        float64
 24  Fwd IAT 

In [23]:
feature_cols = [c for c in df.columns if c != "Label"]

zero_cols_check = [c for c in feature_cols if (df[c] == 0).all()]
print(f"Columns with all zero values: {zero_cols_check}")

near_zero = [c for c in feature_cols if (df[c] == 0).mean() > 0.99]
print(f"\nColumns with 99%+ zero values: {near_zero}")

print("\nZero % for potentially dropped columns:")
for col in near_zero:
    pct = (df[col] == 0).mean() * 100
    print(f"  {col}: {pct:.2f}% zeros")

Columns with all zero values: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']

Columns with 99%+ zero values: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Cnt', 'CWE Flag Count', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']

Zero % for potentially dropped columns:
  Bwd PSH Flags: 100.00% zeros
  Fwd URG Flags: 99.94% zeros
  Bwd URG Flags: 100.00% zeros
  FIN Flag Cnt: 99.67% zeros
  CWE Flag Count: 99.94% zeros
  Fwd Byts/b Avg: 100.00% zeros
  Fwd Pkts/b Avg: 100.00% zeros
  Fwd Blk Rate Avg: 100.00% zeros
  Bwd Byts/b Avg: 100.00% zeros
  Bwd Pkts/b Avg: 100.00% zeros
  Bwd Blk Rate Avg: 100.00% zeros


In [24]:
zero_cols = [c for c in feature_cols if (df[c] == 0).all()]
df.drop(columns=zero_cols, inplace=True)
print(f"Dropped {len(zero_cols)} zero-only columns: {zero_cols}")

Dropped 8 zero-only columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']


In [25]:
# Fix -1 sentinel in window byte columns
win_cols = ["Init Fwd Win Byts", "Init Bwd Win Byts"]
for col in win_cols:
    if col in df.columns:
        df[col] = df[col].replace(-1, 0)
        print(f"Fixed -1 values in '{col}'")

# Drop rows with inf / -inf
inf_mask = df.isin([np.inf, -np.inf]).any(axis=1)
df = df[~inf_mask].copy()
print(f"Dropped {inf_mask.sum()} rows with inf/-inf values")

# Drop rows with any remaining negative feature values
feature_cols = [c for c in df.columns if c != "Label"]
neg_mask = (df[feature_cols] < 0).any(axis=1)
df = df[~neg_mask].copy()
print(f"Dropped {neg_mask.sum()} rows with negative values")

Fixed -1 values in 'Init Fwd Win Byts'
Fixed -1 values in 'Init Bwd Win Byts'
Dropped 10219 rows with inf/-inf values
Dropped 0 rows with negative values


In [26]:
before = len(df)
df.dropna(inplace=True)
df = df[df["Label"].notna()]
df = df[df["Label"].astype(str).str.strip() != ""]
df.reset_index(drop=True, inplace=True)
print(f"Dropped {before - len(df)} rows with NaN/empty Label values")
print(f"Cleaned dataset shape: {df.shape}")
print(f"\nLabel distribution:\n{df['Label'].value_counts()}")
print(f"\nUnique labels: {sorted(df['Label'].unique().tolist())}")

Dropped 0 rows with NaN/empty Label values
Cleaned dataset shape: (2700001, 70)

Label distribution:
Label
Benign                      1743772
DoS attacks-Hulk             461912
Bot                          286191
DoS attacks-SlowHTTPTest     139890
Infilteration                 68236
Name: count, dtype: int64

Unique labels: ['Benign', 'Bot', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'Infilteration']


In [27]:
# Identify binary flag columns (values in {0, 1} only) + categorical cols to skip scaling
flag_cols = [
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags",
    "FIN Flag Cnt", "SYN Flag Cnt", "RST Flag Cnt",
    "PSH Flag Cnt", "ACK Flag Cnt", "URG Flag Cnt",
    "CWE Flag Count", "ECE Flag Cnt",
]
# Keep only the ones still present after zero-column removal
flag_cols = [c for c in flag_cols if c in df.columns]

print("Flag column ranges:")
print(f"{'Column':<25} {'Min':>6} {'Max':>6} Unique values")
print("-" * 60)
for col in flag_cols:
    print(f"{col:<25} {df[col].min():>6.0f} {df[col].max():>6.0f}  {sorted(df[col].unique().tolist())}")

Flag column ranges:
Column                       Min    Max Unique values
------------------------------------------------------------
Fwd PSH Flags                  0      1  [0, 1]
Fwd URG Flags                  0      1  [0, 1]
FIN Flag Cnt                   0      1  [0, 1]
SYN Flag Cnt                   0      1  [0, 1]
RST Flag Cnt                   0      1  [0, 1]
PSH Flag Cnt                   0      1  [0, 1]
ACK Flag Cnt                   0      1  [0, 1]
URG Flag Cnt                   0      1  [0, 1]
CWE Flag Count                 0      1  [0, 1]
ECE Flag Cnt                   0      1  [0, 1]


In [28]:
skip_cols    = flag_cols + ["Label", "Protocol"]
feature_cols = [c for c in df.columns if c not in skip_cols]
print(f"Continuous features to scale : {len(feature_cols)}")
print(f"Skipped (flag/categorical)   : {len(skip_cols)}")

Continuous features to scale : 58
Skipped (flag/categorical)   : 12


In [29]:
print(f"NaN in Label: {df['Label'].isna().sum()}")
print(f"Label types : {df['Label'].apply(type).value_counts().to_dict()}")

NaN in Label: 0
Label types : {<class 'str'>: 2700001}


In [30]:
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print(f"Normalized {len(feature_cols)} continuous features.")
print(f"Skipped    {len(skip_cols)} flag/categorical columns.")
print(f"Final shape: {df.shape}")

Normalized 58 continuous features.
Skipped    12 flag/categorical columns.
Final shape: (2700001, 70)


In [31]:
X = df.drop(columns=["Label"])
y = df["Label"]

print(f"Dataset shape : {X.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"\nDataset is balanced — no oversampling needed.")

Dataset shape : (2700001, 69)

Class distribution:
Label
Benign                      1743772
DoS attacks-Hulk             461912
Bot                          286191
DoS attacks-SlowHTTPTest     139890
Infilteration                 68236
Name: count, dtype: int64

Dataset is balanced — no oversampling needed.


In [32]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train size : {X_train.shape}")
print(f"Val size   : {X_val.shape}")
print(f"Test size  : {X_test.shape}")
print(f"\nTrain label distribution:\n{y_train.value_counts()}")

Train size : (1890000, 69)
Val size   : (405000, 69)
Test size  : (405001, 69)

Train label distribution:
Label
Benign                      1220640
DoS attacks-Hulk             323338
Bot                          200334
DoS attacks-SlowHTTPTest      97923
Infilteration                 47765
Name: count, dtype: int64


In [33]:
train_df = pd.DataFrame(X_train.values, columns=X.columns)
train_df["Label"] = y_train.values

val_df = pd.DataFrame(X_val.values, columns=X.columns)
val_df["Label"] = y_val.values

test_df = pd.DataFrame(X_test.values, columns=X.columns)
test_df["Label"] = y_test.values

print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1890000 entries, 0 to 1889999
Data columns (total 70 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Protocol           float64
 1   Flow Duration      float64
 2   Tot Fwd Pkts       float64
 3   Tot Bwd Pkts       float64
 4   TotLen Fwd Pkts    float64
 5   TotLen Bwd Pkts    float64
 6   Fwd Pkt Len Max    float64
 7   Fwd Pkt Len Min    float64
 8   Fwd Pkt Len Mean   float64
 9   Fwd Pkt Len Std    float64
 10  Bwd Pkt Len Max    float64
 11  Bwd Pkt Len Min    float64
 12  Bwd Pkt Len Mean   float64
 13  Bwd Pkt Len Std    float64
 14  Flow Byts/s        float64
 15  Flow Pkts/s        float64
 16  Flow IAT Mean      float64
 17  Flow IAT Std       float64
 18  Flow IAT Max       float64
 19  Flow IAT Min       float64
 20  Fwd IAT Tot        float64
 21  Fwd IAT Mean       float64
 22  Fwd IAT Std        float64
 23  Fwd IAT Max        float64
 24  Fwd IAT Min        float64
 25  Bwd IAT Tot       

In [34]:
os.makedirs("processed", exist_ok=True)

train_df.to_csv("processed/train.csv", index=False)
val_df.to_csv("processed/val.csv",     index=False)
test_df.to_csv("processed/test.csv",   index=False)

print("Saved:")
print(f"  processed/train.csv  →  {train_df.shape}")
print(f"  processed/val.csv    →  {val_df.shape}")
print(f"  processed/test.csv   →  {test_df.shape}")
print(f"\nNum features: {X_train.shape[1]}")
print(f"Num classes : {y.nunique()} → {sorted(y.unique().tolist())}")

Saved:
  processed/train.csv  →  (1890000, 70)
  processed/val.csv    →  (405000, 70)
  processed/test.csv   →  (405001, 70)

Num features: 69
Num classes : 5 → ['Benign', 'Bot', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'Infilteration']
